# Khai Triển Taylor — Nền Tảng Toán Học của Muon Optimizer

> **Mục tiêu của phần này:** Hiểu tại sao và bằng cách nào ta có thể xấp xỉ sự thay đổi của hàm mất mát $L_i$ khi cập nhật trọng số, dẫn đến công thức Triplet Quadratic Surrogate.

---

## Phần 1 — Công thức Taylor 1 chiều (nền tảng)

Xuất phát từ định lý tích phân cơ bản, với hàm $g: [0,1] \to \mathbb{R}$ bất kỳ:

$$g(1) - g(0) = \int_0^1 g'(t)\, dt$$

Áp dụng tích phân từng phần với $u = g'(t)$ và $dv = dt$, ta triển khai vế phải:

$$g(1) - g(0) = \Big[-g'(t)(1-t)\Big]_0^1 + \int_0^1 (1-t)\,g''(t)\, dt$$

Tính cận: tại $t=1$ thì $(1-t)=0$; tại $t=0$ thì $-(1-0)g'(0) = -g'(0)$. Suy ra:

$$g(1) - g(0) = g'(0) + \int_0^1 (1-t)\,g''(t)\, dt$$

> **Kết quả then chốt (Taylor 1 chiều):**
> $$\boxed{g(1) = g(0) + g'(0) + \int_0^1 (1-t)\,g''(t)\, dt}$$
>
> Đây là Taylor bậc 1 với phần dư tích phân bậc 2 — chính xác hơn so với khai triển bậc 1 thông thường vì phần dư $\int_0^1 (1-t)g''(t)\,dt$ giữ lại **toàn bộ** thông tin độ cong.

---

## Phần 2 — Mã hóa bài toán Deep Learning vào $g(t)$

### Bối cảnh

- $W$: ma trận trọng số hiện tại của một lớp mạng.
- $z_i$: vector đầu vào của mẫu dữ liệu thứ $i$ → đầu ra của lớp là $y = Wz_i$.
- $Q$: **ma trận cập nhật** (lượng ta muốn thay đổi trọng số). Trọng số mới là $W - Q$.
- $L_i$: hàm mất mát trên mẫu thứ $i$.

**Câu hỏi cốt lõi:** $L_i$ thay đổi bao nhiêu khi ta đổi $W \to W - Q$?

### Xây dựng hàm $g(t)$

Thay vì so sánh trực tiếp hai điểm $W$ và $W-Q$ (vốn phức tạp trong không gian nhiều chiều), ta tạo một **đường thẳng tham số** nối chúng:

$$g(t) = L_i\!\left(Wz_i - t\,Qz_i\right), \quad t \in [0, 1]$$

- Tại $t=0$: $g(0) = L_i(Wz_i)$ — mất mát **trước** cập nhật.
- Tại $t=1$: $g(1) = L_i(Wz_i - Qz_i) = L_i((W-Q)z_i)$ — mất mát **sau** cập nhật.

Bài toán đa chiều phức tạp đã được **quy về hàm một biến** $g(t)$. Ta có thể áp dụng ngay công thức Taylor ở Phần 1!

---

## Phần 3 — Tính $g'(0)$ và $g''(t)$ bằng Chain Rule

Đặt vector trạng thái $u(t) = Wz_i - t\,Qz_i$ (kích thước $m \times 1$) để viết gọn:

$$g(t) = L_i(u(t)), \qquad u'(t) = \frac{d}{dt}u(t) = -Qz_i$$

### 3.1 — Đạo hàm bậc 1: $g'(t)$

Áp dụng chain rule:

$$g'(t) = \nabla L_i(u(t))^\top \cdot u'(t) = \nabla L_i(u(t))^\top (-Qz_i)$$

> **Lưu ý chiều:** $\nabla L_i(u(t))$ là vector cột $m \times 1$, chuyển vị thành vector hàng $1 \times m$, nhân với $(-Qz_i)$ kích thước $m \times 1$ → cho ra **scalar** như mong đợi ($g'(t)$ là đạo hàm của một hàm số thực).

Tại $t = 0$:

$$\boxed{g'(0) = -\nabla L_i(Wz_i)^\top (Qz_i)}$$

Đây chính là **tích vô hướng âm** giữa gradient của mất mát và hướng cập nhật — ý nghĩa trực quan: nếu $Q$ đi theo hướng gradient, $g'(0) < 0$, tức mất mát giảm ở điểm khởi đầu.

---

### 3.2 — Đạo hàm bậc 2: $g''(t)$

Lấy đạo hàm của $g'(t)$ theo $t$:

$$g''(t) = \frac{d}{dt}\left[\nabla L_i(u(t))^\top (-Qz_i)\right]$$

Vì $(-Qz_i)$ **không phụ thuộc** $t$, ta chỉ cần lấy đạo hàm phần gradient:

$$g''(t) = \left[\frac{d}{dt}\nabla L_i(u(t))\right]^\top (-Qz_i)$$

**Bước mấu chốt:** đạo hàm của gradient theo $t$ sinh ra ma trận **Hessian**, qua chain rule:

$$\frac{d}{dt}\nabla L_i(u(t)) = \underbrace{\nabla^2 L_i(u(t))}_{m \times m} \cdot \underbrace{u'(t)}_{m \times 1} = \nabla^2 L_i(u(t)) \cdot (-Qz_i)$$

Thay vào biểu thức $g''(t)$:

$$g''(t) = \left[\nabla^2 L_i(u(t)) \cdot (-Qz_i)\right]^\top (-Qz_i)$$

Dùng tính chất chuyển vị $(AB)^\top = B^\top A^\top$:

$$g''(t) = (-Qz_i)^\top \cdot \left[\nabla^2 L_i(u(t))\right]^\top \cdot (-Qz_i)$$

Vì **Hessian luôn đối xứng** ($\nabla^2 L_i = (\nabla^2 L_i)^\top$), và hai dấu âm triệt tiêu nhau $(-1)(-1) = 1$:

$$\boxed{g''(t) = (Qz_i)^\top \cdot \nabla^2 L_i\!\left(Wz_i - tQz_i\right) \cdot (Qz_i)}$$

> **Ý nghĩa hình học:** $g''(t)$ đo **độ cong** của mặt mất mát theo hướng cập nhật $Qz_i$. Nếu Hessian xác định dương ($\nabla^2 L_i \succ 0$), $g''(t) > 0$ — mặt mất mát lồi theo hướng đó.

---

## Phần 4 — Lắp ráp: Khai triển Taylor cho hàm mất mát

Thay $g(0)$, $g'(0)$, $g''(t)$ vào công thức Taylor ở Phần 1:

$$L_i(Wz_i - Qz_i) = L_i(Wz_i) - \nabla L_i(Wz_i)^\top (Qz_i) + \underbrace{\int_0^1 (1-t)\,(Qz_i)^\top \nabla^2 L_i(Wz_i - tQz_i)\,(Qz_i)\,dt}_{\text{phần dư bậc 2}}$$

Vì $(Qz_i)$ không phụ thuộc $t$, ta rút ra ngoài dấu tích phân:

$$\boxed{L_i(Wz_i - Qz_i) = L_i(Wz_i) - \nabla L_i(Wz_i)^\top (Qz_i) + h(Qz_i,\, Wz_i)}$$

trong đó phần dư bậc 2 là:

$$h(Qz_i,\, Wz_i) = (Qz_i)^\top \!\left[\int_0^1 (1-t)\,\nabla^2 L_i(Wz_i - tQz_i)\,dt\right] (Qz_i)$$

---

## Phần 5 — Vấn đề thực tế và phép xấp xỉ xuất sắc

### Tại sao không tính trực tiếp được?

Tích phân $\displaystyle\int_0^1 (1-t)\,\nabla^2 L_i(Wz_i - tQz_i)\,dt$ đòi hỏi:

1. Tính ma trận **Hessian $m \times m$** tại **vô số điểm** dọc theo quỹ đạo $u(t)$.
2. Với mạng thực tế có hàng **tỷ tham số** ($m \sim 10^9$), ma trận Hessian có $\sim 10^{18}$ phần tử — **hoàn toàn bất khả thi**.

### Phép xấp xỉ: "Đóng băng" độ cong

Nhóm tác giả thay thế Hessian phụ thuộc đường đi bằng một **ma trận độ cong trung bình** $H$ **cố định**, không phụ thuộc $t$:

$$\int_0^1 (1-t)\,\nabla^2 L_i(Wz_i - tQz_i)\,dt \;\approx\; H\int_0^1 (1-t)\,dt = H \cdot \frac{1}{2}$$

vì $\displaystyle\int_0^1 (1-t)\,dt = \left[t - \frac{t^2}{2}\right]_0^1 = \frac{1}{2}$.

Phần dư trở thành:

$$h(Qz_i,\, Wz_i) \approx \frac{1}{2}(Qz_i)^\top H\,(Qz_i)$$

> **Đây chính là nguồn gốc của hệ số $\tfrac{1}{2}$** xuất hiện trong công thức Triplet Quadratic Surrogate.

---

## Phần 6 — Từ xấp xỉ đến công thức Triplet Quadratic Surrogate

Tổng hợp trên toàn bộ $N$ mẫu và áp dụng **Trace Trick** để chuyển tích vô hướng thành trace ma trận:

$$(Qz_i)^\top H\,(Qz_i) = \text{tr}\!\left(H\,Qz_iz_i^\top Q^\top\right)$$

Lấy trung bình trên $N$ mẫu:

$$\frac{1}{N}\sum_{i=1}^N (Qz_i)^\top H\,(Qz_i) = \text{tr}\!\left(H\,Q\!\left(\frac{1}{N}\sum_{i=1}^N z_iz_i^\top\right)\!Q^\top\right) = \text{tr}\!\left(H\,Q\,ZZ^\top Q^\top\right)$$

trong đó $ZZ^\top = \frac{1}{N}\sum_i z_iz_i^\top$ là **ma trận tương quan đặc trưng**.

Toàn bộ hàm surrogate (mô hình xấp xỉ) cần tối thiểu hóa theo $Q$ là:

$$\boxed{\mathcal{L}_{\text{surrogate}}(Q) = \underbrace{-\,\text{tr}(G^\top Q)}_{\text{bậc 1 (gradient)}} + \underbrace{\frac{1}{2N}\,\text{tr}\!\left(H\,Q\,ZZ^\top Q^\top\right)}_{\text{bậc 2 (độ cong)}}}$$

trong đó $G = \frac{1}{N}\sum_i \nabla_{W} L_i$ là gradient trung bình.

> **Nhờ việc đóng băng $H$**, bài toán tối ưu hóa phức tạp trên hàm mất mát gốc đã được đưa về bài toán **toán học ma trận thuần túy** — có thể giải bằng phương pháp Newton-Schulz để sinh ra luật cập nhật của Muon.

---

## Tóm tắt luồng suy luận

```
Hàm 1 chiều g(t)          Taylor 1 chiều
g(t) = L_i(Wz_i - tQz_i)  ───────────────►  g(1) = g(0) + g'(0) + ∫(1-t)g''(t)dt
         │
         │  Chain Rule
         ▼
g'(0)  = -∇L_i(Wz_i)ᵀ(Qz_i)          ← tích gradient & hướng cập nhật
g''(t) = (Qz_i)ᵀ ∇²L_i(u(t)) (Qz_i)  ← Hessian đo độ cong
         │
         │  Xấp xỉ: ∇²L_i ≈ H (cố định)
         ▼
Phần dư  ≈  ½ (Qz_i)ᵀ H (Qz_i)       ← nguồn gốc hệ số ½
         │
         │  Trace Trick + tổng N mẫu
         ▼
Surrogate:  -tr(GᵀQ)  +  ½N·tr(HQ·ZZᵀ·Qᵀ)   ← giải bằng Newton-Schulz → Muon
```

# Khai Triển Taylor — Nền Tảng Toán Học của Muon Optimizer

> **Mục tiêu của phần này:** Hiểu tại sao và bằng cách nào ta có thể xấp xỉ sự thay đổi của hàm mất mát $L_i$ khi cập nhật trọng số, dẫn đến công thức Triplet Quadratic Surrogate.

---

## Phần 1 — Công thức Taylor 1 chiều (nền tảng)

Xuất phát từ định lý tích phân cơ bản, với hàm $g: [0,1] \to \mathbb{R}$ bất kỳ:

$$g(1) - g(0) = \int_0^1 g'(t)\, dt$$

Áp dụng tích phân từng phần với $u = g'(t)$ và $dv = dt$, ta triển khai vế phải:

$$g(1) - g(0) = \Big[-g'(t)(1-t)\Big]_0^1 + \int_0^1 (1-t)\,g''(t)\, dt$$

Tính cận: tại $t=1$ thì $(1-t)=0$; tại $t=0$ thì $-(1-0)g'(0) = -g'(0)$. Suy ra:

$$g(1) - g(0) = g'(0) + \int_0^1 (1-t)\,g''(t)\, dt$$

> **Kết quả then chốt (Taylor 1 chiều):**
> $$\boxed{g(1) = g(0) + g'(0) + \int_0^1 (1-t)\,g''(t)\, dt}$$
>
> Đây là Taylor bậc 1 với phần dư tích phân bậc 2 — chính xác hơn so với khai triển bậc 1 thông thường vì phần dư $\int_0^1 (1-t)g''(t)\,dt$ giữ lại **toàn bộ** thông tin độ cong.

---

## Phần 2 — Mã hóa bài toán Deep Learning vào $g(t)$

### Bối cảnh

- $W$: ma trận trọng số hiện tại của một lớp mạng.
- $z_i$: vector đầu vào của mẫu dữ liệu thứ $i$ → đầu ra của lớp là $y = Wz_i$.
- $Q$: **ma trận cập nhật** (lượng ta muốn thay đổi trọng số). Trọng số mới là $W - Q$.
- $L_i$: hàm mất mát trên mẫu thứ $i$.

**Câu hỏi cốt lõi:** $L_i$ thay đổi bao nhiêu khi ta đổi $W \to W - Q$?

### Xây dựng hàm $g(t)$

Thay vì so sánh trực tiếp hai điểm $W$ và $W-Q$ (vốn phức tạp trong không gian nhiều chiều), ta tạo một **đường thẳng tham số** nối chúng:

$$g(t) = L_i\!\left(Wz_i - t\,Qz_i\right), \quad t \in [0, 1]$$

- Tại $t=0$: $g(0) = L_i(Wz_i)$ — mất mát **trước** cập nhật.
- Tại $t=1$: $g(1) = L_i(Wz_i - Qz_i) = L_i((W-Q)z_i)$ — mất mát **sau** cập nhật.

Bài toán đa chiều phức tạp đã được **quy về hàm một biến** $g(t)$. Ta có thể áp dụng ngay công thức Taylor ở Phần 1!

---

## Phần 3 — Tính $g'(0)$ và $g''(t)$ bằng Chain Rule

Đặt vector trạng thái $u(t) = Wz_i - t\,Qz_i$ (kích thước $m \times 1$) để viết gọn:

$$g(t) = L_i(u(t)), \qquad u'(t) = \frac{d}{dt}u(t) = -Qz_i$$

### 3.1 — Đạo hàm bậc 1: $g'(t)$

Áp dụng chain rule:

$$g'(t) = \nabla L_i(u(t))^\top \cdot u'(t) = \nabla L_i(u(t))^\top (-Qz_i)$$

> **Lưu ý chiều:** $\nabla L_i(u(t))$ là vector cột $m \times 1$, chuyển vị thành vector hàng $1 \times m$, nhân với $(-Qz_i)$ kích thước $m \times 1$ → cho ra **scalar** như mong đợi ($g'(t)$ là đạo hàm của một hàm số thực).

Tại $t = 0$:

$$\boxed{g'(0) = -\nabla L_i(Wz_i)^\top (Qz_i)}$$

Đây chính là **tích vô hướng âm** giữa gradient của mất mát và hướng cập nhật — ý nghĩa trực quan: nếu $Q$ đi theo hướng gradient, $g'(0) < 0$, tức mất mát giảm ở điểm khởi đầu.

---

### 3.2 — Đạo hàm bậc 2: $g''(t)$

Lấy đạo hàm của $g'(t)$ theo $t$:

$$g''(t) = \frac{d}{dt}\left[\nabla L_i(u(t))^\top (-Qz_i)\right]$$

Vì $(-Qz_i)$ **không phụ thuộc** $t$, ta chỉ cần lấy đạo hàm phần gradient:

$$g''(t) = \left[\frac{d}{dt}\nabla L_i(u(t))\right]^\top (-Qz_i)$$

**Bước mấu chốt:** đạo hàm của gradient theo $t$ sinh ra ma trận **Hessian**, qua chain rule:

$$\frac{d}{dt}\nabla L_i(u(t)) = \underbrace{\nabla^2 L_i(u(t))}_{m \times m} \cdot \underbrace{u'(t)}_{m \times 1} = \nabla^2 L_i(u(t)) \cdot (-Qz_i)$$

Thay vào biểu thức $g''(t)$:

$$g''(t) = \left[\nabla^2 L_i(u(t)) \cdot (-Qz_i)\right]^\top (-Qz_i)$$

Dùng tính chất chuyển vị $(AB)^\top = B^\top A^\top$:

$$g''(t) = (-Qz_i)^\top \cdot \left[\nabla^2 L_i(u(t))\right]^\top \cdot (-Qz_i)$$

Vì **Hessian luôn đối xứng** ($\nabla^2 L_i = (\nabla^2 L_i)^\top$), và hai dấu âm triệt tiêu nhau $(-1)(-1) = 1$:

$$\boxed{g''(t) = (Qz_i)^\top \cdot \nabla^2 L_i\!\left(Wz_i - tQz_i\right) \cdot (Qz_i)}$$

> **Ý nghĩa hình học:** $g''(t)$ đo **độ cong** của mặt mất mát theo hướng cập nhật $Qz_i$. Nếu Hessian xác định dương ($\nabla^2 L_i \succ 0$), $g''(t) > 0$ — mặt mất mát lồi theo hướng đó.

---

## Phần 4 — Lắp ráp: Khai triển Taylor cho hàm mất mát

Thay $g(0)$, $g'(0)$, $g''(t)$ vào công thức Taylor ở Phần 1:

$$L_i(Wz_i - Qz_i) = L_i(Wz_i) - \nabla L_i(Wz_i)^\top (Qz_i) + \underbrace{\int_0^1 (1-t)\,(Qz_i)^\top \nabla^2 L_i(Wz_i - tQz_i)\,(Qz_i)\,dt}_{\text{phần dư bậc 2}}$$

Vì $(Qz_i)$ không phụ thuộc $t$, ta rút ra ngoài dấu tích phân:

$$\boxed{L_i(Wz_i - Qz_i) = L_i(Wz_i) - \nabla L_i(Wz_i)^\top (Qz_i) + h(Qz_i,\, Wz_i)}$$

trong đó phần dư bậc 2 là:

$$h(Qz_i,\, Wz_i) = (Qz_i)^\top \!\left[\int_0^1 (1-t)\,\nabla^2 L_i(Wz_i - tQz_i)\,dt\right] (Qz_i)$$

---

## Phần 5 — Vấn đề thực tế và phép xấp xỉ xuất sắc

### Tại sao không tính trực tiếp được?

Tích phân $\displaystyle\int_0^1 (1-t)\,\nabla^2 L_i(Wz_i - tQz_i)\,dt$ đòi hỏi:

1. Tính ma trận **Hessian $m \times m$** tại **vô số điểm** dọc theo quỹ đạo $u(t)$.
2. Với mạng thực tế có hàng **tỷ tham số** ($m \sim 10^9$), ma trận Hessian có $\sim 10^{18}$ phần tử — **hoàn toàn bất khả thi**.

### Phép xấp xỉ: "Đóng băng" độ cong

Nhóm tác giả thay thế Hessian phụ thuộc đường đi bằng một **ma trận độ cong trung bình** $H$ **cố định**, không phụ thuộc $t$:

$$\int_0^1 (1-t)\,\nabla^2 L_i(Wz_i - tQz_i)\,dt \;\approx\; H\int_0^1 (1-t)\,dt = H \cdot \frac{1}{2}$$

vì $\displaystyle\int_0^1 (1-t)\,dt = \left[t - \frac{t^2}{2}\right]_0^1 = \frac{1}{2}$.

Phần dư trở thành:

$$h(Qz_i,\, Wz_i) \approx \frac{1}{2}(Qz_i)^\top H\,(Qz_i)$$

> **Đây chính là nguồn gốc của hệ số $\tfrac{1}{2}$** xuất hiện trong công thức Triplet Quadratic Surrogate.

---

## Phần 6 — Lắp ráp toàn cục: Từ $N$ mẫu đơn lẻ đến hàm mất mát batch

### 6.1 — Hàm mất mát toàn cục

Hàm mất mát trên toàn bộ batch $N$ mẫu là trung bình cộng:

$$f(W) = \frac{1}{N}\sum_{i=1}^N L_i(Wz_i)$$

Khi dịch chuyển trọng số một lượng $Q$, **tổng thay đổi mất mát** là:

$$f(W-Q) - f(W) = \frac{1}{N}\sum_{i=1}^N \Big[L_i(Wz_i - Qz_i) - L_i(Wz_i)\Big]$$

### 6.2 — Thay xấp xỉ Taylor vào từng mẫu

Từ kết quả Phần 4 và phép xấp xỉ đóng băng độ cong ở Phần 5, lượng thay đổi của **một mẫu thứ $i$** là:

$$L_i(Wz_i - Qz_i) - L_i(Wz_i) \approx \underbrace{-\nabla L_i(Wz_i)^\top (Qz_i)}_{\text{bậc 1}} + \underbrace{\frac{1}{2}(Qz_i)^\top H\,(Qz_i)}_{\text{bậc 2}}$$

Gộp tổng trên $N$ mẫu:

$$f(W-Q) - f(W) \approx \underbrace{-\frac{1}{N}\sum_{i=1}^N \nabla L_i(Wz_i)^\top (Qz_i)}_{\text{thành phần tuyến tính}} + \underbrace{\frac{1}{2N}\sum_{i=1}^N (Qz_i)^\top H\,(Qz_i)}_{\text{thành phần bậc 2}}$$

> **Bài toán tiếp theo:** Hai dấu tổng $\sum$ cồng kềnh này cần được "gom" thành dạng ma trận gọn gàng. Đây là lúc **Trace Trick** phát huy sức mạnh.

---

## Phần 7 — Ma trận hóa bằng Trace Trick

### Tính chất nền tảng của Trace

Trước khi áp dụng, ta cần hai đẳng thức sau:

| Tính chất | Công thức | Ý nghĩa |
|-----------|-----------|---------|
| Tích vô hướng → Trace | $u^\top v = \text{tr}(vu^\top)$ | Biến scalar thành trace của outer product |
| Dạng toàn phương → Trace | $x^\top Ax = \text{tr}(Axx^\top)$ | Biến quadratic form thành trace |
| Giao hoán vòng | $\text{tr}(ABC) = \text{tr}(BCA) = \text{tr}(CAB)$ | Cho phép đẩy ma trận hằng ra ngoài $\sum$ |

---

### 7.1 — Ma trận hóa thành phần bậc 1 (Linear Term)

Áp dụng tính chất tích vô hướng với $u = \nabla L_i(Wz_i)$ và $v = Qz_i$:

$$\nabla L_i(Wz_i)^\top (Qz_i) = \text{tr}\!\Big((Qz_i)\,\nabla L_i(Wz_i)^\top\Big)$$

Dùng tính chất giao hoán vòng để đẩy $Q$ ra ngoài (vì $Q$ không phụ thuộc $i$):

$$= \text{tr}\!\Big(Q \cdot z_i\,\nabla L_i(Wz_i)^\top\Big)$$

Lấy tổng trên $N$ mẫu, đưa $Q$ và $\text{tr}$ ra ngoài dấu $\sum$:

$$\frac{1}{N}\sum_{i=1}^N \nabla L_i(Wz_i)^\top (Qz_i) = \text{tr}\!\left(Q \underbrace{\left[\frac{1}{N}\sum_{i=1}^N z_i\,\nabla L_i(Wz_i)^\top\right]}_{= G^\top}\right)$$

Đại lượng trong ngoặc vuông chính là **chuyển vị của ma trận gradient toàn cục** $G$:

$$G = \frac{1}{N}\sum_{i=1}^N \nabla L_i(Wz_i)\,z_i^\top \quad \Longrightarrow \quad G^\top = \frac{1}{N}\sum_{i=1}^N z_i\,\nabla L_i(Wz_i)^\top$$

Do đó, thành phần bậc 1 rút gọn thành:

$$\boxed{-\frac{1}{N}\sum_{i=1}^N \nabla L_i(Wz_i)^\top (Qz_i) = -\text{tr}(QG^\top) = -\text{tr}(G^\top Q)}$$

> **Lưu ý:** $\text{tr}(QG^\top) = \text{tr}(G^\top Q)$ do tính giao hoán vòng — đây là dạng **tích vô hướng Frobenius** $\langle G, Q \rangle_F$ giữa hai ma trận.

---

### 7.2 — Ma trận hóa thành phần bậc 2 (Quadratic Term)

Áp dụng tính chất dạng toàn phương với $x = Qz_i$ và $A = H$:

$$(Qz_i)^\top H\,(Qz_i) = \text{tr}\!\Big(H\,(Qz_i)(Qz_i)^\top\Big) = \text{tr}\!\Big(H\,Q\,z_iz_i^\top Q^\top\Big)$$

> **Tại sao $(Qz_i)(Qz_i)^\top = Qz_iz_i^\top Q^\top$?** Đây là tính chất chuyển vị: $(AB)^\top = B^\top A^\top$, áp dụng cho vector $Qz_i$.

Lấy tổng trên $N$ mẫu, vì $H$ và $Q$ không phụ thuộc $i$:

$$\frac{1}{2N}\sum_{i=1}^N (Qz_i)^\top H\,(Qz_i) = \frac{1}{2N}\,\text{tr}\!\left(H\,Q\underbrace{\left[\sum_{i=1}^N z_iz_i^\top\right]}_{= ZZ^\top}Q^\top\right)$$

Đại lượng $\displaystyle\sum_{i=1}^N z_iz_i^\top$ là **ma trận moment bậc hai của đầu vào** — ký hiệu $ZZ^\top$ khi $Z = [z_1\; z_2\; \cdots\; z_N]$ là ma trận ghép các vector đặc trưng theo cột.

$$\boxed{+\frac{1}{2N}\sum_{i=1}^N (Qz_i)^\top H\,(Qz_i) = \frac{1}{2N}\,\text{tr}\!\left(H\,Q\,ZZ^\top Q^\top\right)}$$

> **Ý nghĩa của $ZZ^\top$:** Đây là **ma trận tương quan đặc trưng** (feature covariance), mô tả "hình học" của dữ liệu trong không gian đặc trưng. Nó cho biết các chiều đặc trưng tương quan với nhau như thế nào trong toàn bộ batch.

---

## Phần 8 — Công thức Triplet Quadratic Surrogate hoàn chỉnh

### Kết quả tổng hợp

Cộng hai thành phần từ Phần 7 lại, ta thu được hàm surrogate hoàn chỉnh:

$$\boxed{\mathcal{L}_{\text{surrogate}}(Q) = \underbrace{-\,\text{tr}(G^\top Q)}_{\substack{\text{bậc 1} \\ \text{(hướng gradient)}}} + \underbrace{\frac{1}{2N}\,\text{tr}\!\left(H\,Q\,ZZ^\top Q^\top\right)}_{\substack{\text{bậc 2} \\ \text{(hình phạt độ cong)}}}}$$

### Ý nghĩa của từng thành phần

| Thành phần | Ký hiệu | Vai trò |
|-----------|---------|---------|
| Ma trận gradient | $G$ | Hướng giảm mất mát (giống SGD/Adam) |
| Ma trận độ cong | $H$ | "Đo" mặt cong của hàm mất mát — ngăn bước nhảy quá lớn |
| Ma trận tương quan đặc trưng | $ZZ^\top$ | Hình học của dữ liệu — điều chỉnh theo phân phối đầu vào |

### Tại sao đây là bài toán ma trận thuần túy?

Vì cả ba ma trận $G$, $H$, $ZZ^\top$ đều được coi là **hằng số** (tính một lần từ batch hiện tại), hàm $\mathcal{L}_{\text{surrogate}}(Q)$ là hàm **bậc hai thuần túy theo $Q$**. Lấy đạo hàm theo $Q$ và đặt bằng 0 cho ta điều kiện tối ưu:

$$\frac{\partial \mathcal{L}_{\text{surrogate}}}{\partial Q} = -G + \frac{1}{N}\,H\,Q\,ZZ^\top = 0$$

$$\Longrightarrow \quad H\,Q\,ZZ^\top = N\cdot G$$

Đây là một **phương trình Sylvester ma trận** — có thể giải xấp xỉ cực nhanh bằng **Newton-Schulz iteration**, sinh ra luật cập nhật của Muon.

---

## Tóm tắt luồng suy luận toàn bộ

```
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
BƯỚC 1 — Giải tích 1 chiều
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
g(t) = L_i(Wz_i - tQz_i)   [mã hóa bài toán về hàm 1 biến]
   │
   │  Taylor + Chain Rule
   ▼
g'(0)  = -∇L_i(Wz_i)ᵀ(Qz_i)
g''(t) = (Qz_i)ᵀ ∇²L_i(u(t)) (Qz_i)   [Hessian xuất hiện]
   │
   ▼
L_i(Wz_i - Qz_i) = L_i(Wz_i) - ∇L_iᵀ(Qz_i) + ∫(1-t)g''(t)dt

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
BƯỚC 2 — Xấp xỉ: "Đóng băng" độ cong
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
∇²L_i(u(t))  ──xấp xỉ──►  H  (cố định, không đổi theo t và i)
   │
   ▼
∫(1-t)dt = ½   ──►   Phần dư ≈ ½(Qz_i)ᵀ H (Qz_i)   [nguồn gốc hệ số ½]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
BƯỚC 3 — Tổng hợp N mẫu + Trace Trick → Ma trận hóa
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Bậc 1:  (1/N)Σ ∇L_iᵀ(Qz_i)  ──Trace──►  tr(G^T Q)
                                           G = ma trận gradient toàn cục

Bậc 2:  (1/2N)Σ (Qz_i)ᵀH(Qz_i)  ──Trace──►  (1/2N)·tr(HQ·ZZᵀ·Qᵀ)
                                               ZZᵀ = ma trận tương quan đặc trưng
   │
   ▼
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
KẾT QUẢ — Triplet Quadratic Surrogate
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  L_surrogate(Q) = -tr(GᵀQ)  +  (1/2N)·tr(HQ·ZZᵀ·Qᵀ)
                     ↑                    ↑
               gradient            hình phạt độ cong
                   │
                   │  dL/dQ = 0  →  Phương trình Sylvester
                   ▼
            HQ·ZZᵀ = N·G   ──Newton-Schulz──►   Muon Update
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
```